# ResNet50 Training & Evaluation Notebook
### Project: AI-Based Automated Road-Surface Condition Monitoring Using Vehicle-Mounted Cameras
**Dataset**: RDD2020 (4 Damage Categories: D00 Longitudinal Crack, D10 Transverse Crack, D20 Alligator Crack, D40 Pothole)  
**Model Architecture**: ResNet50 (Pretrained on ImageNet)

## 1. Imports and System Setup

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Add parent directory to sys.path to access src modules
sys.path.append(os.path.abspath(".."))

from src.models.resnet50 import get_resnet50_model, ResNet50RoadClassifier
from src.utils import set_seed, get_device, create_dataloaders, RDD2020_CLASSES
from src.evaluate import evaluate_model, generate_classification_metrics, plot_and_save_confusion_matrix

# Reproducibility & Device Configuration
set_seed(42)
device = get_device()
print(f"Using PyTorch version: {torch.__version__}")

## 2. Configurable Dataset Path & Hyperparameters
> **Note**: Specify the path to your local copy of the RDD2020 dataset directory below.

In [ ]:
# Configurable Dataset Path (Do NOT hardcode fixed machine-specific paths)
DATASET_DIR = "../data/RDD2020"

# Training & Model Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_CLASSES = 4
LEARNING_RATE = 1e-4
EPOCHS = 10
DROPOUT_RATE = 0.2
PRETRAINED = True
FREEZE_BACKBONE = False

CLASS_NAMES = ["D00_Longitudinal_Crack", "D10_Transverse_Crack", "D20_Alligator_Crack", "D40_Pothole"]

print(f"Dataset Directory: {DATASET_DIR}")
print(f"Target Output Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## 3. DataLoaders Setup

In [ ]:
if os.path.exists(DATASET_DIR):
    dataloaders = create_dataloaders(
        data_dir=DATASET_DIR,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        img_size=IMAGE_SIZE
    )
else:
    print(f"[INFO] Dataset directory '{DATASET_DIR}' not found. Please place or point to your dataset directory when ready to run training.")

## 4. ResNet50 Model Architecture Initialization

In [ ]:
model = get_resnet50_model(
    num_classes=NUM_CLASSES,
    pretrained=PRETRAINED,
    freeze_backbone=FREEZE_BACKBONE,
    dropout_rate=DROPOUT_RATE
).to(device)

# Output class verification
dummy_input = torch.randn(2, 3, 224, 224).to(device)
dummy_output = model(dummy_input)

print(f"Model Output Logits Shape: {dummy_output.shape}")
assert dummy_output.shape == (2, NUM_CLASSES), f"Expected shape (2, {NUM_CLASSES}), got {dummy_output.shape}"
print("ResNet50 Architecture adapted successfully for 4 road damage classes.")

## 5. Loss Function & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print("Loss Function: CrossEntropyLoss")
print(f"Optimizer: AdamW (lr={LEARNING_RATE})")

## 6. Output Directories Configuration

In [ ]:
FIGURES_DIR = "../results/figures/resnet50"
MODELS_DIR = "../results/models/resnet50"
TABLES_DIR = "../results/tables/resnet50"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

print(f"Results Output Paths Configured:")
print(f"  - Figures: {FIGURES_DIR}")
print(f"  - Models: {MODELS_DIR}")
print(f"  - Tables: {TABLES_DIR}")